# Quebras temporais — ITS exclusivamente descritivas

Moratória iniciada em julho/2006, Código em maio/2012. Primeiro ano completo 2007/2013, omitindo o ano de transição. Mudança de nível é avaliada no início do período pós; inclinação é a mudança anual. Placebo 2010 é segmentação comparativa, não época garantidamente sem políticas. Fontes oficiais e contexto em LEIA_ME.md.

PPCDAm/DETER, preços, antecipação e outras mudanças impedem atribuição causal. Somente 24 anos, reduzidos a 23 ao omitir transição. HAC lag 1; lag 2 como sensibilidade, t com N−K graus de liberdade. Não há RDD nem DiD.

**Inferência:** se a matriz two-way tiver autovalor negativo, aplica-se projeção espectral (autovalores negativos iguais a zero). SE bruto permanece exportado para auditoria; a correção é determinada pelo diagnóstico, nunca por significância. Sensibilidade municipal/HAC também é salva. A projeção não resolve poucos clusters nem ausência de identificação.

In [1]:
from utilitarios import *
from IPython.display import display
d = ler()
print(f"{len(d)} linhas, {d.id.nunique()} municípios, {d.ano.min()}–{d.ano.max()}")

3384 linhas, 141 municípios, 2001–2024


## 1. Agregar na amostra fixa

Razão de somas, não média simples das taxas. O estoque inicial é endógeno. Pantanal não recebe estimação separada.

In [2]:
series={}
for uso in ['agricultura','pastagem']:
    validos=d.groupby('id')['taxa_'+uso].count().eq(24)
    fixo=d[d.id.isin(validos[validos].index)]
    g=fixo.groupby('ano').agg(ha=('ha_'+uso,'sum'),estoque=('estoque','sum')).reset_index()
    g['taxa_pp']=100*g.ha/g.estoque;g['n_municipios']=fixo.id.nunique()
    series[uso]=g
    g.to_csv(PASTA/f'serie_ITS_{uso}.csv',index=False)
    print(uso,fixo.id.nunique(),'municípios fixos')

agricultura 95 municípios fixos
pastagem 95 municípios fixos


## 2. Moratória — 2007
## 3. Código Florestal — 2013
## 4. Placebo de segmentação — 2010

In [3]:
for uso,g in series.items():
    for ident,corte,transicao in [('T01',2007,2006),('T02',2013,2012),('T03',2010,None)]:
        ident=ident+'_'+uso;s=g[g.ano.ne(transicao)].copy()
        s['tempo']=s.ano-corte;s['pos']=s.ano.ge(corte).astype(int);s['tempo_pos']=s.tempo*s.pos
        formula='taxa_pp ~ tempo + pos + tempo_pos'
        r=smf.ols(formula,s).fit()
        cov=r.get_robustcov_results(cov_type='HAC',maxlags=1,use_correction=True).cov_params()
        display(exportar(ident,f'Quebra descritiva {corte}',formula,s,r,cov,'ITS','nenhum','taxa_pp_'+uso,base_n=24,alerta='HAC lag1; políticas concorrentes; antecipação; sem contrafactual'))
        cov2=r.get_robustcov_results(cov_type='HAC',maxlags=2,use_correction=True).cov_params()
        pd.DataFrame({'id':ident,'termo':r.params.index,'se_HAC_lag2':np.sqrt(np.diag(cov2))}).to_csv(PASTA/f'sensibilidade_{ident}.csv',index=False)
        pd.DataFrame([{'id':ident,'municipios_agregados':int(g.n_municipios.iloc[0]),'anos_pre':int(s.pos.eq(0).sum()),'anos_pos':int(s.pos.eq(1).sum()),'resid_autocorr_lag1':pd.Series(r.resid).autocorr(),'ano_transicao_omitido':transicao}]).to_csv(PASTA/f'diagnostico_{ident}.csv',index=False)

,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,T01_agricultura,Intercept,0.318696,0.099306,0.110847,0.526545,0.004618,23,HAC lag1; políticas concorrentes; antecipação;...
1,T01_agricultura,tempo,0.022989,0.020114,-0.019110,0.065089,0.267267,23,HAC lag1; políticas concorrentes; antecipação;...
2,T01_agricultura,pos,-0.273983,0.097851,-0.478787,-0.069179,0.011424,23,HAC lag1; políticas concorrentes; antecipação;...
3,T01_agricultura,tempo_pos,-0.023529,0.020269,-0.065953,0.018895,0.260090,23,HAC lag1; políticas concorrentes; antecipação;...


,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,T02_agricultura,Intercept,-0.045141,0.041705,-0.132430,0.042147,0.292620,23,HAC lag1; políticas concorrentes; antecipação;...
1,T02_agricultura,tempo,-0.024427,0.008637,-0.042504,-0.006349,0.010743,23,HAC lag1; políticas concorrentes; antecipação;...
2,T02_agricultura,pos,0.110568,0.041809,0.023061,0.198075,0.015985,23,HAC lag1; políticas concorrentes; antecipação;...
3,T02_agricultura,tempo_pos,0.020391,0.008670,0.002244,0.038537,0.029615,23,HAC lag1; políticas concorrentes; antecipação;...


,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,T03_agricultura,Intercept,0.010870,0.042922,-0.078663,0.100404,0.802655,24,HAC lag1; políticas concorrentes; antecipação;...
1,T03_agricultura,tempo,-0.027213,0.011922,-0.052082,-0.002344,0.033536,24,HAC lag1; políticas concorrentes; antecipação;...
2,T03_agricultura,pos,0.039786,0.044334,-0.052692,0.132264,0.380165,24,HAC lag1; políticas concorrentes; antecipação;...
3,T03_agricultura,tempo_pos,0.025836,0.011981,0.000844,0.050829,0.043404,24,HAC lag1; políticas concorrentes; antecipação;...


,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,T01_pastagem,Intercept,1.884573,0.288791,1.280126,2.489020,0.000003,23,HAC lag1; políticas concorrentes; antecipação;...
1,T01_pastagem,tempo,0.103390,0.064332,-0.031257,0.238038,0.124514,23,HAC lag1; políticas concorrentes; antecipação;...
2,T01_pastagem,pos,-1.255006,0.311529,-1.907043,-0.602969,0.000718,23,HAC lag1; políticas concorrentes; antecipação;...
3,T01_pastagem,tempo_pos,-0.098413,0.063700,-0.231739,0.034912,0.138852,23,HAC lag1; políticas concorrentes; antecipação;...


,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,T02_pastagem,Intercept,0.234247,0.188507,-0.160302,0.628797,0.229116,23,HAC lag1; políticas concorrentes; antecipação;...
1,T02_pastagem,tempo,-0.114891,0.035797,-0.189815,-0.039967,0.004615,23,HAC lag1; políticas concorrentes; antecipação;...
2,T02_pastagem,pos,0.475522,0.189844,0.078173,0.872871,0.021521,23,HAC lag1; políticas concorrentes; antecipação;...
3,T02_pastagem,tempo_pos,0.113418,0.037241,0.035472,0.191363,0.006655,23,HAC lag1; políticas concorrentes; antecipação;...


,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,T03_pastagem,Intercept,0.584811,0.194856,0.178349,0.991272,0.007056,24,HAC lag1; políticas concorrentes; antecipação;...
1,T03_pastagem,tempo,-0.114358,0.049790,-0.218218,-0.010499,0.032560,24,HAC lag1; políticas concorrentes; antecipação;...
2,T03_pastagem,pos,0.020218,0.194521,-0.385546,0.425983,0.918254,24,HAC lag1; políticas concorrentes; antecipação;...
3,T03_pastagem,tempo_pos,0.123913,0.051242,0.017025,0.230802,0.025260,24,HAC lag1; políticas concorrentes; antecipação;...
